# Dfns

> Functions and operators written in braces

A dfn is a function written in braces. `⍵` is its right argument:

In [ ]:
{⍵×2}3

6

`⍺` is its left argument:

In [ ]:
2{⍺+⍵}3

5

## Statements

A dfn returns the value of its last statement. Statements are separated by `⋄` or by line breaks:

In [ ]:
{x←⍵+1
 x×2}3

8

The values of earlier statements are discarded:

In [ ]:
{1 ⋄ 2}0

2

A statement can run for its side effect alone, such as adding each item to a total:

In [ ]:
{s←0
 {s+←⍵}¨⍵
 s}1 2 3

6

A dfn that ends in an assignment returns the assigned value without displaying it:

In [ ]:
r←{x←⍵+1}3
r

4

## Scope

Names follow lexical scope. Modified assignment, such as `+←`, updates the nearest binding of its name, looking outward through enclosing dfns to the top level:

In [ ]:
aa←3
bob←{aa+←⍵ ⋄ aa}
bob 5

8

Plain assignment always creates a name local to its dfn:

In [ ]:
{gg←1 ⋄ {gg←⍵}¨⍳⍵ ⋄ gg}5

1

## Bodies and predicates

`;` separates a dfn's bodies. A [predicate](glyphs/question.qmd) `C?` tests the Boolean singleton `C`. When `C` is false, the dfn abandons the body and tries the next one. This dfn negates a negative number:

In [ ]:
{⍵<0?-⍵;⍵}¯3

3

After a predicate, a body can hold several statements:

In [ ]:
{⍵<0?a←-⍵ ⋄ a×2;⍵}¯3

6

A name assigned in one body is still there in the next:

In [ ]:
{a←⍵ ⋄ a>9?"big";a>0?"small";"none"}5

small

Every body but the last needs a predicate. If every predicate fails, the call is a `DOMAIN` error.

An empty body, such as `{}` or the first body of `{⍵≡0?;⍵}`, returns no result.

## Arguments

In a call with one argument, `⍺` is absent. An expression that uses an absent `⍺` is evaluated as if `⍺` weren't there. Called with one argument, `{⍺-⍵}` negates:

In [ ]:
{⍺-⍵}3

¯3

Called with two, the same dfn subtracts:

In [ ]:
10{⍺-⍵}3

7

Anything computed only from an absent `⍺` drops out with it:

In [ ]:
{(2×⍺)-⍵}3

¯3

In a list, an absent `⍺` leaves no item:

In [ ]:
{≢[⍺ ⍵]}3

1ₓ

Any other use, such as `x←⍺` or `⍺` as an operand, is a `VALUE` error.

`⍺←` gives `⍺` a [default](glyphs/alpha.qmd). The assignment runs only when `⍺` is absent:

In [ ]:
sub←{⍺←10 ⋄ ⍺-⍵}
sub 3
2 sub 3

7

¯1

[Valences](glyphs/valences.qmd), `⊘`, joins a monadic function and a dyadic function into one.

## Recursion

[`∇`](glyphs/del.qmd) calls the current dfn. This dfn is factorial:

In [ ]:
{⍵=0?1;⍵×∇⍵-1}5

120

A call to `∇` that is the last thing its body does reuses the current frame.

## Operators

A dfn that uses `⍶`, its left operand, is a monadic operator. `twice` applies its operand two times:

In [ ]:
twice←{⍶ ⍶ ⍵}
(-twice)3

3

A dfn that also uses `⍹`, its right operand, is a dyadic operator. This one applies `⍶` between `⍺` and `⍹ ⍵`:

In [ ]:
2 (+{⍺ ⍶ ⍹ ⍵}-) 3

¯1

[`⍢`](glyphs/del-diaeresis.qmd) refers to the current operator, for recursion with new operands.

## Error guards

An [error guard](glyphs/error-guard.qmd) `codes::handler` makes the dfn return `handler` when one of the listed errors happens after it, in any later statement or body. `÷"a"` raises error 11, `DOMAIN`:

In [ ]:
{11::0 ⋄ ÷⍵}"a"

0

## Design

BPL's dfns come from Dyalog APL, with three rules from BQN's [blocks](https://mlochbaum.github.io/BQN/doc/block.html): the last statement gives the result, predicates choose between bodies separated by `;`, and an absent left argument drops out of the expression, as BQN's `𝕨` does.

In Dyalog APL, a statement that isn't an assignment returns at once. Running a statement only for its side effect needs a workaround.

Marshall Lochbaum's [five-year review of BQN's design](https://mlochbaum.github.io/BQN/commentary/fiveyears.html) criticises two BQN features that BPL leaves out: choosing a body by argument count, which Valences covers in BPL, and block headers. The review finds that BQN's rule for `𝕨`, which BPL adopts, makes ambivalent code easier to write than tacit combinators do.

Unlike the review, BPL still lets special names decide what braces define. Braces whose body uses `⍶` or `⍹` define an operator. Other braces define a function.

Unlike a BQN block, a dfn keeps one scope across its bodies, as Dyalog APL's guards do.